# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and analyzing the FAIR² dataset with the [`mlcroissant`](https://mlcommons.github.io/croissant/) library, referencing all entities by their Croissant `@id`.

### Dataset Source
FAIR² Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Instantiate the mlcroissant Datset
dataset = mlc.Dataset(croissant_url)

# Access metadata fields
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Examine available record sets (`@id`) and their fields. All entities are referenced by their `@id`.

In [ ]:
# Retrieve all record sets and fields by their @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets.\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    # Fields/columns in each record set
    print("  Fields and columns @id:")
    for field in rs.fields:
        print(f"    Field @id: {field.id}, Name: {field.name}, Description: {field.description}")
    print()

## 3. Data Extraction
Load data from record sets into DataFrames for exploration, using `@id` for references. The primary tabular data is usually under the main experiment record set.

In [ ]:
# Collect @id for each record set (adjust depending on your dataset structure)
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Using .records() to load all records for the record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Show columns for first populated DataFrame
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id and main_rs_id in dataframes:
    print(f"Columns for RecordSet @id: {main_rs_id}")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No populated record sets found.")

## 4. Exploratory Data Analysis (EDA)
Explore fields, filter, normalize, and group data using only `@id` columns. For this dataset, possible numeric fields are e.g. 'MW' (Molecular Weight) or 'Abundance'. Replace these by their actual `@id` as collected above.

In [ ]:
# Example: Analyze the 'MW' (Molecular Weight) numeric field using its @id.

# Replace with the actual @id for molecular weight from step 2
numeric_field_id = None
group_field_id = None

# Find likely numeric field and group field (by name heuristics)
if main_rs_id and main_rs_id in dataframes:
    cols = dataframes[main_rs_id].columns.tolist()
    for col in cols:
        if 'weight' in col.lower() or 'mw' in col.lower():
            numeric_field_id = col
        if 'description' in col.lower() or 'group' in col.lower():
            group_field_id = col
    if not numeric_field_id:
        print("Could not identify a numeric field for analysis. Please update 'numeric_field_id'.")
    else:
        threshold = dataframes[main_rs_id][numeric_field_id].mean() if dataframes[main_rs_id][numeric_field_id].dtype != 'O' else 0
        try:
            filtered_df = dataframes[main_rs_id][dataframes[main_rs_id][numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            display(filtered_df.head())
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
            if group_field_id and group_field_id in filtered_df:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
                display(grouped_df.head())
        except Exception as e:
            print(f"Could not perform numeric filtering/normalization: {e}")
else:
    print("No main DataFrame available for numerical analysis.")

## 5. Visualization
Visualize the distribution of the numeric field and relationship with a group/description field, using `@id`-based columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and main_rs_id in dataframes and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[main_rs_id][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group_field_id is available, plot group summary
    if group_field_id and group_field_id in dataframes[main_rs_id]:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=dataframes[main_rs_id][group_field_id], y=dataframes[main_rs_id][numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No available main DataFrame or numeric field to plot.")

## 6. Conclusion
This notebook demonstrated reproducible FAIR data exploration using [`mlcroissant`](https://mlcommons.github.io/croissant/) of the FAIR² mass spectrometry dataset. All entities such as record sets and fields were referenced _only_ by their `@id` in code, ensuring clarity and standardization for downstream analysis.

*You can now extend this notebook for more advanced FAIR analytics (e.g., multi-recordset joins, advanced EDA, or machine learning pipelines), always referencing fields and entities by their Croissant `@id`.*